[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/elastica/elastica.ipynb)

# The Elastica

This example computes the elastica of Euler and Kirchhoff, the shapes elastic rods take under loads at their ends: a strip curled into rings by a couple, a rod pinned at both ends and bent by end thrust into Euler's buckled shapes, and a wire coiled into helices by a couple oblique to it. Each short piece of a rod turns in proportion to the turning effect of the load it carries, and with loads only at the ends that load is the same all along the rod; the shape follows piece by piece from the end where the rod starts.

The computation is written in the projective geometric algebra of three-dimensional space, in which the placement of each cross-section of the rod is a motor and the load is a line: a force along the line, or a couple. Seen from each cross-section, that one line sets how the cross-section turns, and the running product of those turns places the whole rod.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import PGA3D as ga
from examples.animation import save_figure, show_animation
from examples.mechanics.elastica import render
from examples.mechanics.elastica.core import between, points
from examples.mechanics.elastica.scenarios import chords, thrusts

In [ ]:
context = NumpyContext(ga)
mv = context.multivector
Point = ga.gatype.antivector()
Line = ga.gatype.bivector()
Motor = ga.gatype.rotor()
# How a cross-section turns per unit length under the load it carries.
Compliance = ga.gatype((Line, Line))                       # Line <- Line

# The centre of the cross-section at the rod's start, the shift carrying a cross-section along the unbent rod
# per unit length, and the axes of a cross-section through its centre: across, up and along the rod. A turn
# about an axis is the axis itself and the couple about it the axis's dual; a shift along an axis is the dual and
# the force along it the axis.
centre = mv.w.dual()                                       # [] Point
shift_along_rod = mv.z ^ mv.w                              # [] Line
section_axes = stack([centre & mv.x.dual(), centre & mv.y.dual(), centre & mv.z.dual()])   # [3] Line

## 1. A rod as a chain of cross-sections

The rod is modelled as a chain of cross-sections, each a thin rigid slice of the rod with its own placement. The part of the rod beyond any cross-section is held in balance by the load at the far end and by the cross-section alone, so every cross-section carries the same load: one line fixed in space. Seen from a cross-section, `frame << load`, the load's moments about the cross-section's own axes turn it, bending it across and up and twisting it along the rod, each in proportion over the rod's rigidity, its resistance to that turn. Those turns, summed over the axes, are a map from loads to turns per unit length, the rod's compliance.

A cross-section moves to the next by a shift along the rod and its turn, a line read as a motion; over a short piece its exponential is the motor from one cross-section to the next, and the running product of those motors places the whole rod.

In the notation of Kirchhoff's rod theory the turn per unit length is $\boldsymbol{\kappa} = B^{-1}\mathbf{m}$, with $B$ the rigidities and $\mathbf{m}$ the moment carried, and the frames solve $\frac{d}{ds}(R, \mathbf{r}) = (R\,[\boldsymbol{\kappa}]_\times, R\,\mathbf{e}_3)$.

In [ ]:
def compliance(turning: np.ndarray, shifting: np.ndarray) -> Compliance:
    """How a cross-section moves per unit length under a load: about each of its axes `[3]`, a turn by the
    load's couple about the axis times the compliance to turning, and a shift by the load's force along the
    axis times the compliance to shifting; zero to shifting for a rod that only bends and twists."""
    couples = section_axes.dual()                                       # [3] Line
    turns = section_axes * (section_axes & Line) * turning             # [3] Line <- Line
    shifts = couples * (couples & Line) * shifting                     # [3] Line <- Line
    return (turns + shifts).sum(axis=0)                                 # [] Line <- Line


def rod(loads: Line, flex: Compliance, starts: Motor, length: float, pieces: int) -> Motor:
    """The frame of each cross-section of rods of the given length `[..., pieces + 1]`, from the start frames
    along, each rod carrying its load across every cross-section."""
    step = length / pieces

    def frames() -> Iterator[Motor]:
        frame = starts.broadcast_to(np.broadcast_shapes(starts.shape, loads.shape))
        yield frame
        for _ in range(pieces):
            # The load seen from a cross-section turns it, and with the shift moves it along the rod; the turn
            # taken over each piece is the one seen from halfway along it.
            halfway = frame * ((shift_along_rod + flex(frame << loads)) * (step / 4)).exp()   # [...] Motor
            frame = frame * ((shift_along_rod + flex(halfway << loads)) * (step / 2)).exp()
            yield frame
    return stack(list(frames()), axis=-1)                               # [..., pieces + 1] Motor


# The rods of the scenes below: their length, the pieces each is cut into, and their compliances about and along
# the cross-section's axes, across, up and along: to turning, bending across and up and twisting, and none to
# shifting, rods that only bend and twist.
length = 1.0
pieces = 200
turning = np.array([1.0, 1.0, 1.4])
shifting = np.zeros(3)
flex = compliance(turning, shifting)                                            # [] Line <- Line
flex

## 2. Curling under a couple

A couple at the end of a strip, a pure turning effect about the axis across it, looks the same from every cross-section the strip turns about that axis, so every piece turns alike: the strip curls into a circle, and closes into a ring when its length is one circumference, under a couple of the rigidity times a full turn over the length.

In [ ]:
# The couple at the strip's end, in whole turns of its end.
curls = [0.0, 0.25, 0.5, 0.75, 1.0, 2.0]
couples = mv.wx * (2 * np.pi / (turning[0] * length) * np.array(curls))  # [curls] Line
curled = rod(couples, flex, mv.rotor(), length, pieces)                  # [curls, pieces + 1] Motor
save_figure(render.draw_curls(curled >> centre), "elastica_curls");

From straight, darkest, the strip curls through a quarter, a half and three quarters of a turn into a closed ring, and under twice the couple into a double ring, all from its clamp.

## 3. Euler's elastica

A rod pinned at both ends, free to turn at each, is pushed together by a thrust along the line through its pins. The load every cross-section carries is that line of thrust, so the rod, started from its first pin at an angle to the line, bends back towards it; for each angle there is one thrust under which it reaches the line again exactly at its full length, and the far pin sits there. Those thrusts, and the distances between the pins, have closed forms in complete elliptic integrals, computed in `thrusts` and `chords`.

In the notation of the classical elastica the thrust is $P = EI\,(2K(k)/L)^2$ and the distance between the pins $L\,(2E(k)/K(k) - 1)$, with $k = \sin(\alpha/2)$ for the angle $\alpha$ at the pins and $K$, $E$ the complete elliptic integrals of the first and second kind.

In [ ]:
# The angles at which the rod leaves its first pin, in degrees, for the plate and for the frames of the sweep,
# and the duration of a frame.
end_angles = [20.0, 60.0, 90.0, 110.0, 130.7, 150.0, 165.0, 175.0]
sweep = np.linspace(2.0, 176.0, 60)
duration_ms = 60


def turned(degrees: np.ndarray) -> Motor:
    """The turns by each angle about the axis across the rod's start."""
    return (mv.yz * (-np.radians(degrees) / 2)).exp()


def euler(degrees: np.ndarray) -> Motor:
    """The frames of rods leaving their first pin at each angle to the line of thrust `[angles, pieces + 1]`,
    under the thrust that brings them back to the line at their full length."""
    thrust = (centre & mv.z.dual()) * thrusts(np.radians(degrees))        # [angles] Line
    return rod(thrust, flex, turned(degrees), length, pieces)


pinned = euler(np.array(end_angles))                                     # [angles, pieces + 1] Motor
save_figure(render.draw_plate(pinned >> centre), "elastica_euler")
show_animation(render.animate_sweep(euler(sweep) >> centre), "elastica_sweep", duration_ms)

At a small angle the rod bows into a shallow arch. As the angle grows the arch deepens, its sides turn upright at a right angle, and past it the pins move closer than the arch is wide; at about 130.7° they meet, Euler's figure eight, and beyond it the rod crosses itself in a loop that tightens as the angle approaches a half turn.

## 4. Kirchhoff's helices

A couple oblique to a wire, a turning effect about an axis tilted from the wire's direction, bends the wire about that axis and twists it along itself. With the same rigidity against bending in every direction, the wire's direction turns about the couple's axis at one rate all along it, and the wire coils into a helix around that axis: tight when the wire starts nearly square to the couple, stretched when it starts nearly along it. A ribbon across the cross-sections shows their twist, the rod being less rigid against twisting than against bending.

In [ ]:
# The tilt of each wire's start from the upward couple, in degrees, the whole turns the couple coils it through,
# and the width of the ribbon drawn along it.
tilts = [80.0, 60.0, 40.0, 20.0]
coils = 3
ribbon = 0.015
couple = mv.wz * (2 * np.pi * coils / (turning[0] * length))           # [] Line
coiled = rod(couple, flex, turned(np.array(tilts)), length, pieces)      # [tilts, pieces + 1] Motor
edges = coiled[..., None] >> points([[-ribbon / 2, 0, 0], [ribbon / 2, 0, 0]])   # [tilts, pieces + 1, 2] Point
save_figure(render.draw_helices(coiled >> centre, edges), "elastica_helices");

From left to right the wires start at 80°, 60°, 40° and 20° from the couple's axis, upward: each coils through three turns around it, the more stretched the closer it starts to the axis.

## 5. From ring to twist

One couple, held fixed, does two things to a wire: its part square to the wire bends it, and its part along the wire twists it. Sweeping the tilt of the wire's start from square to the couple to along it, and back, moves the couple from one to the other. Square to it, the couple only bends: the wire closes into a flat ring and the ribbon lies flat. As the tilt shrinks the ring opens into a helix that stretches out, and the ribbon starts to wring. Along the couple, the wire is straight and only twisted, the ribbon wrung through more turns than the ring had, since the rod is less rigid against twisting than against bending. Each frame is the wire's shape at rest under that tilt, while the camera turns slowly around it.

In [ ]:
# The whole turns the couple coils the wire through, the frames of the sweep, the width of the ribbon, and the
# duration of a frame.
uncoil_turns, uncoil_frames, uncoil_ribbon, uncoil_ms = 1.5, 90, 0.035, 60
# The tilt of the wire's start from the couple, in degrees, from square to along it and back.
sweep_tilts = 45 * (1 + np.cos(np.linspace(0, 2 * np.pi, uncoil_frames, endpoint=False)))   # [frames]
uncoiled = rod(mv.wz * (2 * np.pi * uncoil_turns / (turning[0] * length)), flex, turned(sweep_tilts), length, pieces)
uncoiled_edges = uncoiled[..., None] >> points([[-uncoil_ribbon / 2, 0, 0], [uncoil_ribbon / 2, 0, 0]])   # [frames, pieces + 1, 2] Point
show_animation(render.animate_uncoiling(uncoiled >> centre, uncoiled_edges), "elastica_uncoiling", uncoil_ms)

## 6. A telephone cord, pulled

A telephone cord is a rod whose rest shape is already a coil: its turn per unit length at rest bends and twists it into a helix. In motion it is modelled as a chain of short rigid segments with mass, each joined to the next by an elastic joint whose rest placement is the coil. A joint's deviation from rest is a line, a small turn and shift; its stiffness maps that line to a load, couples for the turns and forces for the shifts. The load a joint puts on its two segments has to be exactly the derivative of the joint's energy, or the motion gains energy from nowhere: it is the stiffness applied to the deviation, corrected by its commutators with the deviation, the derivative of the exponential.

Each segment moves by Euler's equations for a rigid body: its momentum, the inertia applied to its rate, changes by the forque on it, the loads of its two joints, its weight and some damping, less the gyroscopic turn of the momentum by the rate; its frame then moves by the rate over a short time step. The cord hangs from its base, its handset is pulled out along the coil's axis, held, and let back.

In [ ]:
# The load a joint carries per unit of its deviation from rest, and a segment's momentum per unit of rate.
Stiffness = ga.gatype((Line, Line))                        # Line <- Line
Inertia = ga.gatype((Line, Line))                          # Line <- Line


def stiffness(turning: np.ndarray, shifting: np.ndarray) -> Stiffness:
    """The load a joint carries per unit length of its deviation from rest: about each of the cross-section's
    axes `[3]`, a couple for the turn about it times the rigidity against turning, and a force along it for the
    shift along it times the rigidity against shifting."""
    couples = section_axes.dual()                                       # [3] Line
    turns = couples * (couples & Line) * turning                       # [3] Line <- Line
    shifts = section_axes * (section_axes & Line) * shifting           # [3] Line <- Line
    return (turns + shifts).sum(axis=0)                                 # [] Line <- Line


def segment_inertia(mass: float, length: float, radius: float) -> Inertia:
    """A segment's inertia, from its rate to its momentum: its mass spread over its two ends and a ring of the
    given radius around its middle, each point's momentum its join with its own motion under the rate."""
    spread = points([[radius, 0, 0], [-radius, 0, 0], [0, radius, 0], [0, -radius, 0],
                     [0, 0, length / 2], [0, 0, -length / 2]])            # [6] Point
    return (spread & spread.commutator(Line)).sum(axis=0) * (mass / spread.shape[0])   # [] Line <- Line


def joint_loads(frames: Motor, rest: Motor, stiff: Stiffness, step: float) -> Line:
    """The load each joint between neighbouring frames puts on the frame after it, in the world: its stiffness
    applied to its deviation from rest, through the derivative of the exponential."""
    deviation = (rest.reverse() * frames[..., :-1].reverse() * frames[..., 1:]).log() * 2   # [..., joints] Line
    load = stiff(deviation / step)
    load = load - deviation.commutator(load) / 2 + deviation.commutator(deviation.commutator(load)) / 12
    return frames[..., 1:] >> load                                      # [..., joints] Line


def moving(frames: Motor, rates: Line, rest: Motor, stiff: Stiffness, step: float, inertia: Inertia,
           inverse: Inertia, weight: Point, damping: float, time_step: float) -> tuple[Motor, Line]:
    """One time step of the segments between the first and last frame, which are held."""
    loads = joint_loads(frames, rest, stiff, step)                      # [joints] Line
    segments = frames[1:-1]                                             # [segments] Motor
    momenta = inertia(rates)                                            # [segments] Line
    # The weight is a line straight down through the segment's centre; the damping resists its rate.
    forques = (segments << (loads[:-1] - loads[1:])) + (centre & (segments << weight)) - damping * momenta
    rates = rates + inverse(forques - momenta.commutator(rates)) * time_step
    segments = segments * (rates * (-time_step / 2)).exp()
    return concatenate([frames[:1], segments, frames[-1:]]), rates

In [ ]:
# The cord: its length in metres, its coils, the segments per coil, and the angle of its wire to the plane of a
# coil; its rigidities against turning, bending and twisting, in newton square metres, and against shifting,
# shearing and stretching, in newtons; its mass per metre in kilograms, the radius its segments' mass is spread
# over as a share of a segment's length, gravity in metres per second squared, and its damping per second.
cord_length, coils, per_coil, cord_pitch = 1.0, 5, 6, 10.0
cord_turning = np.array([1e-3, 1e-3, 7e-4])
cord_shifting = np.array([20.0, 20.0, 20.0])
cord_mass, cord_spread, gravity, cord_damping = 0.05, 2.0, 9.81, 1.0
# The pull: the time the cord hangs first, the time the handset takes out and the distance, the time it is held,
# and the time it takes back, in seconds and metres; the time step, the steps per frame, the frames, the drawn
# frames between neighbouring segments, the width of the drawn ribbon, and the duration of a frame.
hang, out_time, reach, hold, back_time = 0.3, 0.9, 0.6, 0.3, 0.4
time_step, steps_per_frame, cord_frames = 1e-3, 40, 70
drawn_between, cord_ribbon, cord_ms = 4, 0.025, 40

pieces_of_cord = coils * per_coil
segment = cord_length / pieces_of_cord
pitch = np.radians(cord_pitch)
# At rest the cord shifts along itself while bending and twisting at the rate of its coils.
coiling = shift_along_rod + (mv.yz * np.cos(pitch) + mv.xy * np.sin(pitch)) * (2 * np.pi * coils / cord_length)   # [] Line
rest_frames = (coiling * (np.arange(-1, pieces_of_cord + 1) * segment / 2)).exp()   # [pieces + 2] Motor
joint_rest = (coiling * (segment / 2)).exp()                              # [] Motor
cord_stiffness = stiffness(cord_turning, cord_shifting)
inertia = segment_inertia(cord_mass * segment, segment, cord_spread * segment)   # [] Line <- Line
inverse = inertia.inverse()                                              # [] Line <- Line
weight = mv.y.dual() * (-gravity * cord_mass * segment)                  # [] Point
# The coil's axis is the direction of its turn; the handset moves along it, keeping its turn.
coil_axis = mv.x * np.cos(pitch) + mv.z * np.sin(pitch)                  # [] Vector


def handset_reach(time: float) -> float:
    """How far the handset is out at a time: eased out after the hang, held, and eased back."""
    out = np.clip((time - hang) / out_time, 0, 1)
    back = np.clip((time - hang - out_time - hold) / back_time, 0, 1)
    return reach * (3 * out ** 2 - 2 * out ** 3) * (1 - (3 * back ** 2 - 2 * back ** 3))


def pulled() -> Iterator[Motor]:
    """The cord's frames at every frame of the scene."""
    frames, rates = rest_frames, mv.bivector(np.zeros((pieces_of_cord, 6)))
    for index in range(cord_frames):
        yield frames
        for tick in range(steps_per_frame):
            shift = (coil_axis ^ mv.w) * (handset_reach((index * steps_per_frame + tick) * time_step) / 2)
            frames = concatenate([frames[:-1], (shift.exp() * rest_frames[-1])[None]])
            frames, rates = moving(frames, rates, joint_rest, cord_stiffness, segment, inertia, inverse, weight,
                                   cord_damping, time_step)


moved = stack(list(pulled()))                                            # [frames, pieces + 2] Motor
drawn = between(moved, drawn_between)                                    # [frames, drawn] Motor
drawn_edges = drawn[..., None] >> points([[-cord_ribbon / 2, 0, 0], [cord_ribbon / 2, 0, 0]])   # [frames, drawn, 2] Point
show_animation(render.animate_cord(drawn >> centre, drawn_edges), "elastica_cord", cord_ms)

The cord first sags under its own weight, then stretches out as the handset is pulled, its coils opening into a long even helix; held, it settles, and let back it springs in, its coils tumbling over one another before the damping calms them.

In [ ]:
# checks
# Curled by whole turns, the strip's end comes back to its start.
ends = render.coordinates(curled[[0, 4, 5], -1] >> centre)
assert np.abs(ends[1:]).max() < 1e-11 and abs(ends[0, 2] - length) < 1e-11
# Euler's elastica reach the line of thrust again at their far ends, a chord away from the first pin; the
# tolerance tests the steps along the rod.
far = render.coordinates(pinned[:, -1] >> centre)                         # [angles, 3]
assert np.abs(far[:, :2]).max() < 1e-3
assert np.abs(far[:, 2] - chords(np.radians(end_angles))).max() < 1e-3
# Coiled through whole turns, each wire's end lies straight above its start, the length times the cosine of the
# tilt higher; the tolerance tests the steps along the rod.
tips = render.coordinates(coiled[:, -1] >> centre)                        # [tilts, 3]
assert np.abs(tips - length * np.cos(np.radians(tilts))[:, None] * [0, 0, 1]).max() < 3e-3
# The cord bends and twists but hardly stretches or shears: its joints' shifts stay within a few hundredths of rest,
# the most where the handset springs back.
deviations = (joint_rest.reverse() * moved[:, :-1].reverse() * moved[:, 1:]).log() * 2      # [frames, joints] Line
assert np.abs(deviations.cast(ga.subspace("wx wy wz")).kernel / segment).max() < 0.1